## Assignment: Object detection
- Alumno 1:
- Alumno 2:
- Alumno 3:

The goals of the assignment are:
* Put into practice acquired knowledge to detect and recognize objects of interest within a satellite image.

To address this problem, you must choose one of the following options:
*	Implement a sliding window strategy to process the whole image, and then train a classifier that determines whether each window includes or not an object of interest. In this way, you can use previous image classification model to infer the object category.
*	Build a single-stage object detection model (e.g., YOLO, SSD, RetinaNet, etc.).
*	Build a two-stage object detection model (e.g., Faster R-CNN, R-FCN, etc.).

Follow the link below to download the detection data set “xview_detection”: [https://drive.upm.es/s/P7nEf3Bygns7tbM](https://drive.upm.es/s/P7nEf3Bygns7tbM)

In [1]:
# All imports consolidated here
import torch
from ultralytics import YOLO
import uuid
import numpy as np
import json
import os
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.cuda.is_available()}")

PyTorch version: 2.7.1+cu118
CUDA version: True


In [2]:
class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([])

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [3]:
categories = {0: 'Small car', 1: 'Bus', 2: 'Truck', 3: 'Building'}

In [4]:
# === CONFIGURATION - HPC paths ===
BASE_PATH = '/home/x467/PROJECT/xview_detection'  # Read-only dataset location
WORK_DIR = '/home/x467/x467028'  # Writable working directory
DEVICE = '0'  # GPU device

#### Training
Design and train a detector to deal with the “xview_detection” perception task.

##### Hyperparameters 

Optimization techniques: ADAM and Batch Size

In [5]:
BATCH_SIZE = 4
LEARNING_RATE = 1e-3

CONF = 0.25  # confidence threshold
IOU = 0.5  # IoU threshold for mAP calculation

EPOCHS = 100
SIZE = 640
BALANCE_MULTIPLIER = 1.5  # controls class balancing (1.0 = no oversampling, 2.0 = aggressive)

In [6]:
# Load database
json_file = f'{BASE_PATH}/xview_det_train.json'
with open(json_file) as ifs:
    json_data = json.load(ifs)
ifs.close()

In [7]:
# 1. OPTIMIZATION: Group annotations by image_id first
# This prevents checking every annotation for every image (the previous bottleneck)
ann_dict = defaultdict(list)
for ann in json_data['annotations'].values():
    ann_dict[ann['image_id']].append(ann)

counts = dict.fromkeys(categories.values(), 0)
anns = []

# 2. Iterate through images
for json_img in json_data['images'].values():
    image = GenericImage(json_img['filename'])
    image.tile = np.array([0, 0, json_img['width'], json_img['height']])
    
    # 3. Fast lookup of annotations for the current image
    current_anns = ann_dict.get(json_img['image_id'], [])
    
    for json_ann in current_anns:
        obj = GenericObject()
        # Use image_id (as in original) or ann_id depending on preference, 
        # but maintaining original logic here:
        obj.id = json_ann['image_id'] 
        obj.bb = (int(json_ann['bbox'][0]), int(json_ann['bbox'][1]), int(json_ann['bbox'][2]), int(json_ann['bbox'][3]))
        obj.category = json_ann['category_id']
        
        # Update counts and add to image
        counts[obj.category] += 1
        image.add_object(obj)
        
    anns.append(image)

print(counts)

{'Small car': 188300, 'Bus': 6269, 'Truck': 10600, 'Building': 275943}


In [8]:
# Analyze class distribution
class_counts = Counter()
for img in anns:
    for obj in img.objects:
        class_counts[obj.category] += 1

print("lass Distribution")
total_instances = sum(class_counts.values())
for category_name in categories.values():
    count = class_counts[category_name]
    percentage = (count / total_instances) * 100
    print(f"{category_name}: {count} instances ({percentage:.2f}%)")
print(f"\nTotal instances: {total_instances}")

lass Distribution
Small car: 188300 instances (39.14%)
Bus: 6269 instances (1.30%)
Truck: 10600 instances (2.20%)
Building: 275943 instances (57.36%)

Total instances: 481112


In [9]:
anns_train, anns_valid = train_test_split(anns, test_size=0.1, random_state=1, shuffle=True)
print(f'Training images: {len(anns_train)}')
print(f'Validation images: {len(anns_valid)}')

Training images: 6845
Validation images: 761


In [10]:
# Balance training dataset with controlled oversampling
def balance_dataset(anns, target_multiplier):
    """
    Balance dataset by oversampling images with underrepresented classes
    """
    # Count instances per class and map images to dominant class
    class_counts = Counter()
    image_class_map = []
    
    for img in anns:
        img_class_counts = Counter()
        for obj in img.objects:
            img_class_counts[obj.category] += 1
        
        if img_class_counts:
            dominant_class = img_class_counts.most_common(1)[0][0]
            image_class_map.append((img, dominant_class))
            class_counts[dominant_class] += 1
    
    # Calculate sampling weights
    max_count = max(class_counts.values())
    balanced_anns = []
    
    for img, dominant_class in image_class_map:
        class_count = class_counts[dominant_class]
        repeat_times = int((max_count / class_count) * target_multiplier)
        repeat_times = max(1, repeat_times)
        
        for _ in range(repeat_times):
            balanced_anns.append(img)
    
    print(f"Original: {len(anns)} images")
    print(f"Balanced: {len(balanced_anns)} images")
    
    # Show new distribution
    new_class_counts = Counter()
    for img in balanced_anns:
        for obj in img.objects:
            new_class_counts[obj.category] += 1
    
    total = sum(new_class_counts.values())
    for category_name in categories.values():
        count = new_class_counts[category_name]
        percentage = (count / total) * 100
        print(f"{category_name}: {count} instances ({percentage:.2f}%)")
    
    return balanced_anns

# Apply balancing
anns_train = balance_dataset(anns_train, BALANCE_MULTIPLIER)

print(f'\nFinal training images: {len(anns_train)}')
print(f'Validation images: {len(anns_valid)}')

Original: 6845 images
Balanced: 21494 images


Small car: 357388 instances (40.24%)
Bus: 110270 instances (12.41%)
Truck: 87886 instances (9.89%)
Building: 332658 instances (37.45%)

Final training images: 21494
Validation images: 761


In [11]:
# Create YOLO format labels in the WRITABLE working directory
def create_yolo_dataset(anns, split_name):
    """Convert annotations to YOLO format
    YOLO format: class_id x_center y_center width height (all normalized 0-1)
    """
    # Create labels in the writable working directory
    labels_dir = Path(f'{WORK_DIR}/datasets/{split_name}/labels')
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    # Create images directory with symlink to read-only images
    images_dir = Path(f'{WORK_DIR}/datasets/{split_name}/images')
    images_dir.parent.mkdir(parents=True, exist_ok=True)
    
    # Create symlink to images (if not exists)
    if not images_dir.exists():
        images_dir.symlink_to(f'{BASE_PATH}/xview_train', target_is_directory=True)
        print(f'Created symlink: {images_dir} -> {BASE_PATH}/xview_train')
    
    for img_ann in anns:
        # Create label file for this image
        label_path = labels_dir / (Path(img_ann.filename).stem + '.txt')
        
        with open(label_path, 'w') as f:
            img_width = img_ann.tile[2] - img_ann.tile[0]
            img_height = img_ann.tile[3] - img_ann.tile[1]
            
            for obj in img_ann.objects:
                # Get class id (0-based index)
                class_id = list(categories.keys())[list(categories.values()).index(obj.category)]
                
                # Convert from xyxy to xywh (normalized)
                x1, y1, x2, y2 = obj.bb
                x_center = ((x1 + x2) / 2) / img_width
                y_center = ((y1 + y2) / 2) / img_height
                width = (x2 - x1) / img_width
                height = (y2 - y1) / img_height
                
                # Write to file
                f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
    
    return len(anns)

# Create datasets
print('Creating YOLO format datasets...')
num_train = create_yolo_dataset(anns_train, 'train')
num_val = create_yolo_dataset(anns_valid, 'val')
print(f'Created {num_train} training labels and {num_val} validation labels')

# Create data.yaml configuration file in YOLO format
# This format tells YOLO to look for train/images and train/labels subdirectories
data_yaml = f"""path: {WORK_DIR}/datasets
train: train
val: val

names:
  0: Small car
  1: Bus
  2: Truck
  3: Building
"""

yaml_path = f'{WORK_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(data_yaml)

print('Created data.yaml configuration file')
print(f'Dataset structure:')
print(f'  {WORK_DIR}/datasets/train/images/ -> {BASE_PATH}/xview_train/ (symlink)')
print(f'  {WORK_DIR}/datasets/train/labels/ (created)')
print(f'  {WORK_DIR}/datasets/val/images/ -> {BASE_PATH}/xview_train/ (symlink)')
print(f'  {WORK_DIR}/datasets/val/labels/ (created)')
print(f'Config file: {yaml_path}')

Creating YOLO format datasets...
Created symlink: /home/x467/x467028/datasets/train/images -> /home/x467/PROJECT/xview_detection/xview_train


Created symlink: /home/x467/x467028/datasets/val/images -> /home/x467/PROJECT/xview_detection/xview_train


Created 21494 training labels and 761 validation labels
Created data.yaml configuration file
Dataset structure:
  /home/x467/x467028/datasets/train/images/ -> /home/x467/PROJECT/xview_detection/xview_train/ (symlink)
  /home/x467/x467028/datasets/train/labels/ (created)
  /home/x467/x467028/datasets/val/images/ -> /home/x467/PROJECT/xview_detection/xview_train/ (symlink)
  /home/x467/x467028/datasets/val/labels/ (created)
Config file: /home/x467/x467028/data.yaml


In [12]:
# Initialize YOLOv11 model (medium)
model = YOLO('yolo11m.pt')
print("Model loaded!")

Model loaded!


In [13]:
print('Training YOLOv11 model with balanced classes')

# Train the model
results = model.train(
    data=f'{WORK_DIR}/data.yaml',      # dataset configuration file
    epochs=EPOCHS,                      # number of training epochs
    imgsz=SIZE,                         # input image size (640x640)
    batch=BATCH_SIZE,                   # batch size for training
    lr0=LEARNING_RATE,                  # initial learning rate
    device=DEVICE,                      # GPU device ID
    project=WORK_DIR,                   # directory to save results
    name='yolov11_test',                # experiment name
    exist_ok=True,                      # overwrite existing experiment
    verbose=False,                      # suppress progress bars
    
    cls=1.0,                             # increased from default 0.5 to help with imbalanced classes

    # Basic augmentation
    mosaic=1.0,                         # mosaic augmentation probability
    fliplr=0.5,                         # horizontal flip probability

    # Training optimizations
    patience=15,                        # early stopping patience (epochs)
    save=True,                          # save model checkpoints
    cache=False,                        # disable caching to reduce memory usage
    
    # Validation
    val=True,                           # run validation during training
    plots=True,                         # generate training plots
)

print('Training complete!')
print(f'Model saved at: {WORK_DIR}/yolov11_test/weights/best.pt')

Training YOLOv11 model with balanced classes


Ultralytics 8.3.252 🚀 Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA A100-PCIE-40GB, 40338MiB)


engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/x467/x467028/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov11_test, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=/home/x467/x467028, rect=False, res

Overriding model.yaml nc=80 with nc=4



                   from  n    params  module                                       arguments                     


  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 


... (56,827 outputs left out)

                   all       7592      47247     0.0605      0.401     0.0456      0.023


Speed: 0.1ms preprocess, 2.2ms inference, 0.0ms loss, 1.0ms postprocess per image


Results saved to /media/beegfs/home/x467/x467028/yolov11_test


Training complete!
Model saved at: /home/x467/x467028/yolov11_test/weights/best.pt


#### Validation
Compute validation metrics.

In [14]:
# Load the trained model
model = YOLO(f'{WORK_DIR}/yolov11_test/weights/best.pt')

# Run validation
print('Running validation...')
metrics = model.val(
    data=f'{WORK_DIR}/data.yaml',
    imgsz=SIZE,
    batch=1,
    conf=CONF,
    iou=IOU,
    device=DEVICE,
    plots=True,
    verbose=False,
)

Running validation...
Ultralytics 8.3.252 🚀 Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA A100-PCIE-40GB, 40338MiB)


YOLO11m summary (fused): 125 layers, 20,033,116 parameters, 0 gradients, 67.7 GFLOPs


val: Fast image access ✅ (ping: 0.9±0.2 ms, read: 550.8±12.2 MB/s, size: 1201.3 KB)


val: Scanning /media/beegfs/home/x467/x467028/datasets/val/labels.cache... 761 images, 6845 backgrounds, 14 corrupt: 100% ━━━━━━━━━━━━ 7606/7606 1.5Git/s 0.0s

val: /media/beegfs/home/x467/x467028/datasets/val/images/1076_bc045223-1b8d-4cd4-aa06-990ab63018e8.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.69375   1.8859375]
val: /media/beegfs/home/x467/x467028/datasets/val/images/1086_95cbc6a6-c679-4858-940d-5e131886aca6.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.3125  1.71875]
val: /media/beegfs/home/x467/x467028/datasets/val/images/1094_be9743f8-4fb2-4cb8-a016-4d7655da6d1f.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.078125]
val: /media/beegfs/home/x467/x467028/datasets/val/images/1109_02f1df70-b2bd-4088-b27a-7e9640e29acf.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.078125 1.4875  ]
val: /media/beegfs/home/x467/x467028/datasets/val/images/1430_60dd9f27-66db-41b5-b8d1-91bd4a894d44.tif: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.7171875]
val: /media/beegfs/home/x46

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 2/7592 4.5it/s 0.1s<28:10

... (752 outputs left out)

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7592/7592 96.4it/s 1:19

                   all       7592      47247     0.0612      0.415     0.0486     0.0271


Speed: 0.7ms preprocess, 7.3ms inference, 0.0ms loss, 0.7ms postprocess per image


Results saved to /media/beegfs/home/x467/x467028/runs/detect/val


In [15]:
# Print metrics
print('\n=== Validation Metrics ===')
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

# Per-class metrics
print('\n=== Per-Class Metrics ===')
for i, category in categories.items():
    print(f"{category}:")
    print(f"  AP50: {metrics.box.ap50[i]:.3f}")
    print(f"  Precision: {metrics.box.p[i]:.3f}")
    print(f"  Recall: {metrics.box.r[i]:.3f}")


=== Validation Metrics ===
mAP50: 0.049
mAP50-95: 0.027
Precision: 0.061
Recall: 0.415

=== Per-Class Metrics ===
Small car:
  AP50: 0.062
  Precision: 0.070
  Recall: 0.693
Bus:
  AP50: 0.040
  Precision: 0.058
  Recall: 0.283
Truck:
  AP50: 0.030
  Precision: 0.048
  Recall: 0.145
Building:
  AP50: 0.063
  Precision: 0.069
  Recall: 0.537


#### Testing
Try to improve the results provided in the competition.

In [16]:
anns = []
test_path = os.path.join(BASE_PATH, 'xview_test')

for (dirpath, dirnames, filenames) in os.walk(test_path):
    for filename in filenames:
        if filename.endswith(('.jpg', '.png', '.jpeg', '.tif', '.tiff')):  # only image files
            rel_path = os.path.relpath(dirpath, BASE_PATH)
            image = GenericImage(os.path.join(rel_path, filename))
            image.tile = np.array([0, 0, SIZE, SIZE])
            anns.append(image)
            
print(f'Number of testing images: {len(anns)}')

Number of testing images: 852


In [17]:
# Load the trained model
model = YOLO(f'{WORK_DIR}/yolov11_test/weights/best.pt')

# Process test images
predictions_data = {"images": {}, "annotations": {}, "categories": {}}
predictions_data["categories"] = categories

imgs_idx, annos_idx = 0, 0

for ann in anns:
    # Run inference
    img_path = os.path.join(BASE_PATH, ann.filename)
    results = model.predict(
        source=img_path,
        imgsz=SIZE,
        conf=CONF,
        iou=IOU,
        device=DEVICE,
        verbose=False,
    )[0]
    
    # Extract predictions
    boxes = results.boxes
    num_objects = len(boxes)
    
    # Add image info
    image_data = {
        "image_id": ann.filename.split('/')[-1],
        "filename": ann.filename,
        "num_objects": num_objects,
        "width": SIZE,
        "height": SIZE
    }
    predictions_data["images"][imgs_idx] = image_data
    imgs_idx += 1
    
    # Add annotations
    for i in range(num_objects):
        box = boxes[i]
        xyxy = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].cpu().numpy())
        cls = int(box.cls[0].cpu().numpy())
        
        annotation_data = {
            "image_id": ann.filename.split('/')[-1],
            "category_id": categories[cls],
            "bbox": (int(xyxy[0]), int(xyxy[1]), int(xyxy[2]), int(xyxy[3])),
            "confidence": str(conf)
        }
        predictions_data["annotations"][annos_idx] = annotation_data
        annos_idx += 1

print(f'Processed {len(anns)} test images')
print(f'Total detections: {annos_idx}')

Processed 852 test images
Total detections: 57041


In [18]:
# Save predictions to writable directory
output_path = f'{WORK_DIR}/prediction.json'
with open(output_path, "w") as outfile:
    json.dump(predictions_data, outfile)
    
print(f'Predictions saved to: {output_path}')

Predictions saved to: /home/x467/x467028/prediction.json
